# Copia pública del 22 de octubre de 2026

Código de investigación; revisar entradas en `../README.md`. Las salidas guardadas se retiraron. Las etapas 01–06 no se reejecutaron para esta publicación. Para la síntesis numérica utilizada en la memoria, ejecutar `07_Sintesis_resultados_20261022.ipynb`. Las utilidades interactivas son opcionales de revisión y no una dependencia de la síntesis científica.

In [ ]:
from pathlib import Path
import sys
BASE = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'Configuracion.json').is_file()), None)
if BASE is None:
    raise FileNotFoundError('Iniciar Jupyter en notebooks/actual_20261022 o su subcarpeta de notebooks.')
sys.path.insert(0, str(BASE))
import publicacion_config


# 04 · ksn y fallas
Mapas de ksn local, perfiles completos, ajustes bajo KP con residuos, relaciones espaciales y comparación a lo largo del rumbo de Carranza y Pelluhue. El swath interactivo está centralizado en 03_K_U_tau_20261019.ipynb.

**Preparado sin ejecutar.** Ejecutar 01 primero. Comparte θ, A₀ y ventana con los otros notebooks; el ksn local se lee de la exportación de 01. El ksn de tramo se obtiene de χ–z con intercepto libre y excluye el nodo del KP. Son dos medidas diferentes.

Las asociaciones río–falla se completan solo después de revisar la geometría; la lista vacía conserva el pendiente, sin asignar fallas automáticamente. Los cruces cartográficos y los cambios de ksn no demuestran por sí solos actividad tectónica.

In [ ]:
from pathlib import Path
import sys
BASE = publicacion_config.base()
if not (BASE/'Configuracion.json').is_file():
    BASE = Path(str(BASE / 'inputs_external' / 'Jupyter_Memoria'))
if not (BASE/'Configuracion.json').is_file():
    raise FileNotFoundError('Abrir Jupyter desde la carpeta Jupyter_Memoria.')
CARPETA_NOTEBOOKS = BASE
CARPETA_CODIGO = BASE/'Archivos_Memoria/Codigo'
sys.path.insert(0,str(CARPETA_CODIGO))
import importlib
import memoria_comun as pa
pa=importlib.reload(pa)
CFG,RUN_ID=pa.CFG,pa.RUN_ID
sys.path.insert(0,str(Path(CFG['topoanalysis_codigo']).parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'pdf.fonttype':42,'font.family':'DejaVu Sans','font.size':10})
print('Corrida compartida:',RUN_ID,'| θ:',CFG['theta'],'| ventana:',CFG['ventana_m'],'m')

pa.NOTEBOOK_PATH=BASE/'Notebooks_principales_20261019/04_ksn_Fallas.ipynb'

## 1 Parámetros

θ y ventana proceden de Configuracion.json y de las sensibilidades exportadas por 01. Para una combinación científica nueva, regenerar la exportación compatible en 01 y después ejecutar este notebook. Cambiar sólo una variable gráfica aquí no recalcula las pendientes guardadas. Si cambian únicamente los límites del tramo diagnóstico, ejecutar desde la sección 6.

`None` muestra todos los ríos en los paneles; una lista limita solo las figuras. Los límites de color se comparten entre mapas y paneles. El percentil superior 100 permite distinguir los valores altos que antes quedaban saturados.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util
import json
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from shapely.geometry import Point
from IPython.display import display

CARPETA_NOTEBOOKS = BASE
MODULO_VISUAL = CARPETA_CODIGO/'swath_graben_3d.py'


MODULO_GRAFICOS = CARPETA_CODIGO/'visualizacion_ksn_20261013.py'

# Cálculo del ksn.
THETA_MAPA_KSN = CFG['theta']
VENTANA_MAPA_KSN_M = CFG['ventana_m']
MIN_NODOS_MAPA_KSN = CFG['min_puntos']
A0_GEORGE = CFG['A0_m2']
MIN_NODOS_GEORGE = 10
MIN_LONGITUD_GEORGE_M = 500.0
TOLERANCIA_XY_M = 0.10
TOLERANCIA_Z_M = 0.10

# Figuras: selección independiente del cálculo de los 18 ríos.
RIOS_MAPA = None
RIOS_PANELES = None
RIOS_DETALLE = ['3', '4', '5', '6', '8', '9', '10']
MOSTRAR_MAULE = True
MOSTRAR_MAPA_BAJO_KP = True
MOSTRAR_PANELES_PLANTA = True
MOSTRAR_PERFILES_COMPLETOS = True
MOSTRAR_DETALLES = True
MOSTRAR_COMPARACION_TRAMOS = True
PERCENTILES_COLOR = (2, 100)
LIMITES_COLOR = None  # Opcional: (1.0, 200.0).
MARGEN_ZOOM_KP_M = 2500.0

# Límites revisados, en metros desde el outlet recortado existente.
# Clave (ID, id_kp); valor (inicio, fin). Fin no puede superar el KP.
LIMITES_TRAMOS_M = {}

# Asociaciones justificadas en planta; no se elige automáticamente la falla cercana.
# Ejemplo de formato, sin activar:
# {'ID':'3', 'id_kp':'ID_EXACTO_DEL_CSV', 'falla':'Carranza',
#  'x_ref_m':665000.0, 'y_ref_m':6100000.0, 'motivo':'Cruce o referencia revisada'}
ASOCIACIONES_REVISADAS = []
AZIMUT_RUMBO_GRADOS = {'Carranza':65.0, 'Pelluhue':65.0}

GUARDAR_PRODUCTOS = True
GUARDAR_FIGURAS = True
CARPETA_PRODUCTOS = BASE/'Productos/04_ksn_Fallas'

# Anexo opcional: relieve del polígono guardado.
EJECUTAR_SWATH = False
RUTA_POLIGONO_SWATH = Path(CFG['graben_shp'])
EXAGERACION_VERTICAL_SWATH = 1.0

In [ ]:
def cargar_modulo(nombre, ruta):
    if not ruta.is_file():
        raise FileNotFoundError(f'No existe el módulo: {ruta}')
    spec = importlib.util.spec_from_file_location(nombre, ruta)
    modulo = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = modulo
    spec.loader.exec_module(modulo)
    return modulo

gf_pa = pa
sw = cargar_modulo('swath_perfiles_fallas_20261013', MODULO_VISUAL)
vf = cargar_modulo('graficos_perfiles_fallas_20261013', MODULO_GRAFICOS)
if not (np.isfinite(THETA_MAPA_KSN) and THETA_MAPA_KSN >= 0):
    raise ValueError('Theta debe ser finito y no negativo.')
if not (np.isfinite(VENTANA_MAPA_KSN_M) and VENTANA_MAPA_KSN_M > 0):
    raise ValueError('La ventana debe ser positiva y finita.')
if MIN_NODOS_MAPA_KSN < 3 or MIN_NODOS_GEORGE < 3 or not np.isfinite(A0_GEORGE) or A0_GEORGE <= 0:
    raise ValueError('Revisar el mínimo de nodos y A0.')
print('Entradas de la corrida:', gf_pa.RUN_ID)
print(f'Este análisis: theta={THETA_MAPA_KSN:g}; ventana={VENTANA_MAPA_KSN_M:g} m.')

print('Las figuras usan los módulos guardados; no hay que pegar celdas externas.')

## Entradas y KP manuales
Perfiles de la misma corrida, conservando el outlet existente. θ y ventana se leen de Configuracion.json. La reubicación histórica exige un nodo único en XY y elevación dentro de 0,10 m y conserva las huellas originales y actuales. Los incompatibles detienen el avance.

In [ ]:
gf_perfiles, gf_nombres, gf_manifest = gf_pa.load_profiles()
gf_ids_mapa = sorted(gf_perfiles, key=int) if RIOS_MAPA is None else list(map(str, RIOS_MAPA))
if not gf_ids_mapa or set(gf_ids_mapa) - set(gf_perfiles):
    raise ValueError('Revisar RIOS_MAPA.')
display(pd.DataFrame([
    {'ID':rid, 'rio':gf_nombres[rid], 'nodos':len(p),
     'longitud_km':float(p.distancia_m.iloc[-1])/1000}
    for rid,p in sorted(gf_perfiles.items(),key=lambda item:int(item[0]))
]))

In [ ]:
gf_originales = pd.read_csv(
    Path(gf_pa.CFG["kp_manual_csv"]),
    sep=gf_pa.CFG["kp_separador"],
    decimal=gf_pa.CFG["kp_decimal"],
    dtype={"ID": str},
).dropna(how="all")

gf_originales.columns = gf_originales.columns.str.strip()

gf_necesarias = {
    "ID", "id_kp", "indice", "perfil_hash",
    "distancia_m", "elevacion_m", "x_utm", "y_utm"
}
if gf_necesarias - set(gf_originales.columns):
    raise ValueError(
        f"Faltan columnas: {gf_necesarias - set(gf_originales.columns)}"
    )

gf_aceptados, gf_descartados, gf_vistos = [], [], set()

for _, gf_r in gf_originales.iterrows():
    gf_id_kp = gf_pa.text(gf_r["id_kp"])
    gf_registro = {
        "ID": gf_pa.text(gf_r["ID"]),
        "id_kp": gf_id_kp
    }

    # Una fila sin selección, como Chanco, no es un KP.
    if not gf_id_kp:
        gf_campos_punto = [
            "indice", "distancia_m", "elevacion_m", "x_utm", "y_utm"
        ]
        gf_vacia = gf_r[gf_campos_punto].isna().all()

        gf_registro.update(
            estado="sin_seleccion" if gf_vacia else "incompatible",
            detalle=(
                "Sin KP seleccionado" if gf_vacia
                else "Datos de punto sin id_kp"
            )
        )
        gf_descartados.append(gf_registro)
        continue

    try:
        if gf_id_kp in gf_vistos:
            raise ValueError("id_kp duplicado en el CSV.")
        gf_vistos.add(gf_id_kp)

        gf_id = gf_pa.normalize_id(gf_r["ID"])
        gf_p = gf_perfiles[gf_id]

        gf_xy_original = gf_r[["x_utm", "y_utm"]].to_numpy(float)
        gf_z_original = float(gf_r["elevacion_m"])

        if (
            not np.isfinite(gf_xy_original).all()
            or not np.isfinite(gf_z_original)
        ):
            raise ValueError("Coordenadas o elevación original no finitas.")

        # Buscar un único nodo coincidente dentro de 0,10 m.
        gf_distancias = np.linalg.norm(
            gf_p[["x_utm", "y_utm"]].to_numpy(float) - gf_xy_original,
            axis=1
        )
        gf_coincidencias = np.flatnonzero(gf_distancias <= TOLERANCIA_XY_M)

        if len(gf_coincidencias) != 1:
            raise ValueError(
                f"Coincidencias XY: {len(gf_coincidencias)}; "
                f"distancia mínima: {gf_distancias.min():.3f} m."
            )

        gf_j = int(gf_coincidencias[0])
        gf_nodo = gf_p.iloc[gf_j]

        gf_dz = abs(
            float(gf_nodo["elevacion_m"]) - gf_z_original
        )
        if not np.isfinite(gf_dz) or gf_dz > TOLERANCIA_Z_M:
            raise ValueError(
                f"Desajuste de elevación: {gf_dz:.3f} m."
            )

        gf_aceptados.append({
            "ID": gf_id,
            "id_kp": gf_id_kp,
            "indice": gf_j,
            "distancia_m": float(gf_nodo["distancia_m"]),
            "elevacion_m": float(gf_nodo["elevacion_m"]),
            "x_utm": float(gf_nodo["x_utm"]),
            "y_utm": float(gf_nodo["y_utm"]),
            "indice_original": gf_r["indice"],
            "distancia_original_m": gf_r["distancia_m"],
            "hash_original": gf_r["perfil_hash"],
            "hash_actual": gf_pa.profile_hash(gf_p),
            "desajuste_xy_m": float(gf_distancias[gf_j]),
            "desajuste_z_m": gf_dz,
            "estado": ("ok" if str(gf_r["perfil_hash"]) == gf_pa.profile_hash(gf_p) else "manual_reubicado"),
        })

    except (ValueError, TypeError, KeyError) as gf_error:
        gf_registro.update(
            estado="incompatible",
            detalle=str(gf_error)
        )
        gf_descartados.append(gf_registro)

gf_manuales = pd.DataFrame(gf_aceptados, columns=["ID", "id_kp", "indice", "distancia_m", "elevacion_m", "x_utm", "y_utm", "indice_original", "distancia_original_m", "hash_original", "hash_actual", "desajuste_xy_m", "desajuste_z_m", "estado"])
gf_rechazados = pd.DataFrame(gf_descartados)

if not gf_rechazados.empty:
    display(gf_rechazados)

    if gf_rechazados["estado"].eq("incompatible").any():
        raise ValueError(
            "Hay desajustes reales: revisar la tabla anterior."
        )

if gf_manuales.empty:
    print("Sin KP compatibles: el mapa local puede revisarse; no habrá ajustes bajo KP.")

display(gf_manuales[
    ["ID", "id_kp", "indice", "desajuste_xy_m", "desajuste_z_m"]
].round(6))

print(f"KP manuales verificados: {len(gf_manuales)}")

In [ ]:
def leer_trazas(ruta):
    capa = gpd.read_file(ruta)
    if capa.empty or capa.crs is None:
        raise ValueError(f'Capa vacía o sin CRS: {ruta}')
    if (capa.geometry.isna().any() or capa.geometry.is_empty.any()
        or not capa.geometry.is_valid.all()
        or not capa.geom_type.isin(['LineString','MultiLineString']).all()):
        raise ValueError(f'Revisar geometrías: {ruta}')
    return capa.to_crs(gf_pa.CFG['crs'])

gf_chaf = leer_trazas(Path(gf_pa.CFG['chaf']))
if not {'falla','f_id'}.issubset(gf_chaf.columns):
    raise ValueError('CHAF debe contener falla y f_id.')
gf_chaf['familia'] = gf_chaf.falla.fillna('').astype(str).str.replace('*','',regex=False).str.strip()
gf_trazas_mapa = gf_chaf.loc[gf_chaf.familia.isin(['Carranza','Pelluhue'])].copy()
if gf_trazas_mapa.empty:
    raise ValueError('No se encontraron trazas de Carranza o Pelluhue.')
gf_trazas_mapa['id_traza'] = 'CHAF:' + gf_trazas_mapa.f_id.astype(str)
gf_maule_mapa = leer_trazas(Path(gf_pa.CFG['maule'])) if MOSTRAR_MAULE else None
display(gf_trazas_mapa.reindex(columns=['id_traza','falla','dir_buz','sentido']))

## 3 Calcular el ksn local

En cada nodo se ajusta elevación contra distancia dentro de una ventana centrada. Después:

\[ k_{sn,local}=S_{local}\,A^{\theta}. \]

Se excluyen huecos de elevación, pendientes no positivas y áreas inválidas. Cada segmento recibe el promedio del ksn de sus extremos; si un extremo es inválido, el segmento no recibe un valor. No se reemplazan pendientes negativas por su valor absoluto.

Se registran ventanas parciales y ventanas que mezclan ambos lados de un KP. Estas marcas ayudan a revisar el suavizado; no se eliminan del mapa sin criterio. Los valores regionales originales se distinguen por nombre en la copia de trabajo.

In [ ]:
gf_ksn_perfiles, gf_ksn_segmentos, gf_ksn_qc = {}, {}, []
for rid,p in sorted(gf_perfiles.items(), key=lambda item:int(item[0])):
    q = p.copy()  # ksn y pendiente ya calculados y verificados en 01
    q = q.rename(columns={'ksn':'ksn_regional', 'chi_m':'chi_regional_m'})
    slope, area = q.pendiente.to_numpy(float), q.area_m2.to_numpy(float)
    valid = np.isfinite(slope) & (slope>0) & np.isfinite(area) & (area>0)
    # Nombre mantenido por compatibilidad con el módulo visual; attrs guarda theta real.
    q['ksn_040'] = np.where(valid, slope*area**THETA_MAPA_KSN, np.nan)
    q['ksn_local'] = q['ksn_040']
    q.attrs.update(theta_ksn_mapa=THETA_MAPA_KSN)
    s = q.distancia_m.to_numpy(float)
    cruza = np.zeros(len(q), dtype=bool)
    for skp in gf_manuales.loc[gf_manuales.ID.eq(rid),'distancia_m']:
        cruza |= (s-VENTANA_MAPA_KSN_M/2 < skp) & (s+VENTANA_MAPA_KSN_M/2 > skp)
    q['ventana_cruza_KP'] = cruza
    xy, k = q[['x_utm','y_utm']].to_numpy(float), q.ksn_040.to_numpy(float)
    parcial = q.ventana_parcial.to_numpy(bool)
    gf_ksn_perfiles[rid] = q
    gf_ksn_segmentos[rid] = {
        'xy':np.stack([xy[:-1],xy[1:]],axis=1), 'ksn_040':(k[:-1]+k[1:])/2,
        's0_m':s[:-1], 's1_m':s[1:],
        'ventana_parcial':parcial[:-1] | parcial[1:],
        'ventana_cruza_KP':cruza[:-1] | cruza[1:],
    }
    gf_ksn_qc.append({'ID':rid,'rio':gf_nombres[rid], 'nodos':len(q),
        'nodos_con_ksn':int(valid.sum()), 'ventanas_parciales':int(parcial.sum()),
        'ventanas_cruzan_KP':int(cruza.sum())})
gf_tabla_qc = pd.DataFrame(gf_ksn_qc)
display(gf_tabla_qc)
print(f'ksn local: theta={THETA_MAPA_KSN:g}, unidades m^{2*THETA_MAPA_KSN:g}.')

In [ ]:
vf_vista = vf.preparar(sw, gf_ksn_perfiles, gf_ksn_segmentos,
    theta=THETA_MAPA_KSN, ventana_m=VENTANA_MAPA_KSN_M,
    percentiles=PERCENTILES_COLOR, limites=LIMITES_COLOR)
print(f"Color común: {vf_vista['vmin']:.3g} a {vf_vista['vmax']:.3g} m^{2*THETA_MAPA_KSN:g}.")
print('Los límites de color no alteran los ksn; el cursor muestra el valor calculado.')

## 4 Mapas en planta

Mapa completo: observar dónde cambia el ksn. Mapa bajo KP: localizar los sectores que revisaremos para la comparación entre métodos. Ambos conservan la misma escala. En el segundo, el resto del cauce y los ríos sin KP quedan grises; los colores siguen siendo ksn local.

Menú para vista general, sectores y ríos; rueda para zoom; arrastrar para mover. El cursor muestra ksn, distancia y si la ventana es parcial o cruza el KP. Las trazas conservan su procedencia cartográfica. Si un río tiene varios KP, el mapa bajo KP colorea el dominio aguas abajo del más alejado del outlet.

In [ ]:
ksni_fig = vf.mapa(sw, vf_vista, gf_ksn_perfiles, gf_ksn_segmentos,
    gf_manuales, gf_nombres, gf_trazas_mapa, gf_maule_mapa,
    ruta_dem=gf_pa.CFG['dem'], ids=gf_ids_mapa)
ksni_fig.update_layout(meta={'theta':THETA_MAPA_KSN, 'ventana_m':VENTANA_MAPA_KSN_M,
    'rios':gf_ids_mapa, 'mostrar_maule':MOSTRAR_MAULE})
ksni_fig.show(renderer='browser',config={'scrollZoom':True,'displaylogo':False})
vf_mapa_bajo_kp = None

In [ ]:
if MOSTRAR_MAPA_BAJO_KP and not gf_manuales.empty:
    vf_mapa_bajo_kp = vf.mapa(sw, vf_vista, gf_ksn_perfiles, gf_ksn_segmentos,
        gf_manuales, gf_nombres, gf_trazas_mapa, gf_maule_mapa,
        ruta_dem=gf_pa.CFG['dem'], ids=gf_ids_mapa, solo_bajo_kp=True)
    vf_mapa_bajo_kp.show(renderer='browser',config={'scrollZoom':True,'displaylogo':False})

In [ ]:
if MOSTRAR_PANELES_PLANTA:
    vf.paneles_planta(vf_vista, ksni_fig, gf_ksn_perfiles, gf_ksn_segmentos,
        gf_manuales, gf_nombres, gf_trazas_mapa, gf_maule_mapa,
        ids=RIOS_PANELES, solo_bajo_kp=False)

## 5 Revisar los perfiles completos

Comparar elevación y ksn local a lo largo de cada cauce. Negro: KP; gris: ventanas que cruzan el KP; violeta: cruce XY con una traza cartográfica. Un cruce no demuestra que la falla cause el cambio de pendiente. Las superposiciones lineales no se fuerzan a un punto único.

Aquí se observa si el sector bajo KP es uniforme o tiene varios cambios. Consultar litología y aluvión en el notebook correspondiente antes de introducir límites nuevos. Las curvas conservan sus escalas verticales indicadas.

In [ ]:
sw.validar_ksn(gf_ksn_perfiles,VENTANA_MAPA_KSN_M,THETA_MAPA_KSN)
if MOSTRAR_PERFILES_COMPLETOS:
    vf.paneles_perfiles(vf_vista, gf_ksn_perfiles, gf_manuales, gf_nombres,
        ids=RIOS_PANELES, trazas=gf_trazas_mapa, maule=gf_maule_mapa)

## 6 Calcular el ksn representativo bajo cada KP

El ajuste inicial va desde el outlet existente hasta el nodo anterior al KP; el KP queda fuera. Un límite diferente debe venir de revisión geomorfológica, no de buscar una recta con mayor R². Registrar los cambios en `LIMITES_TRAMOS_M`.

Se recalcula χ con θ y A₀ de esta sección y se ajusta elevación con intercepto libre:

\[ z=b+M_{\chi}\,\chi,\qquad k_{sn,tramo}=M_{\chi}\,A_0^{\theta}. \]

Este valor resume el tramo y no tiene que coincidir con cada ksn local. Revisar la longitud, los residuos y la geología. Se muestran límites solicitados y nodos efectivamente utilizados. Las salidas quedan como «ajuste calculado: revisar»; no se convierten en tasas de levantamiento ni en edades.

In [ ]:
gf_filas, gf_ajustes = [], {}
for rid,p in sorted(gf_perfiles.items(),key=lambda item:int(item[0])):
    kps = gf_manuales.loc[gf_manuales.ID.eq(rid)]
    if kps.empty:
        gf_filas.append({'ID':rid,'rio':gf_nombres[rid],'id_kp':'','estado':'sin KP manual'})
        continue
    chi = gf_pa.chi_1d(p,theta=THETA_MAPA_KSN,A0=A0_GEORGE)
    s, z = p.distancia_m.to_numpy(float), p.elevacion_m.to_numpy(float)
    for kp in kps.itertuples():
        j = int(kp.indice)
        inicio,fin = LIMITES_TRAMOS_M.get((rid,kp.id_kp),(s[0],s[j]))
        fila = {'ID':rid,'rio':gf_nombres[rid],'id_kp':kp.id_kp,
            'theta':THETA_MAPA_KSN,'A0_m2':A0_GEORGE,'perfil_hash':gf_pa.profile_hash(p),
            'inicio_solicitado_m':inicio,'fin_solicitado_m':fin,'elevacion_KP_m':kp.elevacion_m}
        if not np.isfinite([inicio,fin]).all() or not s[0] <= inicio < fin <= s[j]:
            fila['estado'] = 'límites inválidos'
            gf_filas.append(fila);continue
        mask = (np.arange(len(p))<j) & (s>=inicio) & (s<=fin)
        indices = np.flatnonzero(mask)
        longitud = float(s[indices[-1]]-s[indices[0]]) if len(indices)>1 else 0.0
        fila.update(n_nodos=len(indices),longitud_ajuste_m=longitud)
        if len(indices)<MIN_NODOS_GEORGE or longitud<MIN_LONGITUD_GEORGE_M:
            fila['estado'] = 'tramo insuficiente'
        elif not np.isfinite(z[mask]).all():
            fila['estado'] = 'huecos: revisar'
        else:
            fit = linregress(chi[mask],z[mask])
            residuos = z[mask]-(fit.intercept+fit.slope*chi[mask])
            positivo = np.isfinite(fit.slope) and fit.slope>0
            fila.update(estado='ajuste calculado: revisar' if positivo else 'pendiente no positiva',
                Mchi=float(fit.slope),intercepto_m=float(fit.intercept),
                ksn_040=float(fit.slope*A0_GEORGE**THETA_MAPA_KSN) if positivo else np.nan,
                R2=float(fit.rvalue**2),RMSE_m=float(np.sqrt(np.mean(residuos**2))),
                inicio_real_m=float(s[indices[0]]),fin_real_m=float(s[indices[-1]]))
            gf_ajustes[(rid,kp.id_kp)] = dict(p=p,chi=chi,j=j,mask=mask,fit=fit,residuos=residuos)
        gf_filas.append(fila)
tabla_ksn_bajo_KP = pd.DataFrame(gf_filas)
gf_firma_ajustes = gf_pa.signature({'theta':THETA_MAPA_KSN,'A0_m2':A0_GEORGE,
    'limites':{str(k):list(v) for k,v in LIMITES_TRAMOS_M.items()},
    'min_nodos':MIN_NODOS_GEORGE,'min_longitud_m':MIN_LONGITUD_GEORGE_M})
display(tabla_ksn_bajo_KP.reindex(columns=['ID','rio','id_kp','estado','ksn_040','R2','RMSE_m',
    'n_nodos','longitud_ajuste_m','inicio_real_m','fin_real_m']).round(3))
print('Intercepto libre; KP excluido. No se calculan K, U ni tiempos.')

In [ ]:
if MOSTRAR_DETALLES:
    vf.detalle(vf_vista, gf_ksn_perfiles, gf_manuales, gf_nombres,
        gf_ajustes, tabla_ksn_bajo_KP, ids=RIOS_DETALLE,
        margen_kp_m=MARGEN_ZOOM_KP_M, trazas=gf_trazas_mapa, maule=gf_maule_mapa)

In [ ]:
if MOSTRAR_COMPARACION_TRAMOS:
    vf.comparar(vf_vista,tabla_ksn_bajo_KP,ids=RIOS_PANELES)

## 7 Comparar los tramos a lo largo del rumbo de las fallas

Completar `ASOCIACIONES_REVISADAS` con el ID de río, ID del KP, falla y una referencia espacial justificada en planta. Puede ser un cruce o un punto del tramo revisado. No se utiliza automáticamente la falla cercana ni el KP como vínculo tectónico.

El eje NE 65° es una proyección exploratoria definida por familia. Su cero es el extremo SW de las trazas disponibles. Se conservan los alcances de cada registro: no se unen discontinuidades ni se interpreta la suma como longitud de falla. Cambiar el azimut o las asociaciones requiere repetir esta sección.

Si la lista está vacía, esta comparación permanece pendiente. Ríos y tramos con varias selecciones no son observaciones independientes. Primero se describe el patrón; su interpretación requiere controles geológicos y cartográficos.

In [ ]:
columnas = ['falla','ID','rio','id_kp','distancia_rumbo_km','ksn_040','R2','RMSE_m',
            'x_ref_m','y_ref_m','distancia_a_traza_km','motivo']
gf_comparacion = pd.DataFrame(columns=columnas)
gf_alcances = []
if not ASOCIACIONES_REVISADAS:
    print('Pendiente: revisar el mapa y completar ASOCIACIONES_REVISADAS en la sección 1.')
else:
    firma_actual = gf_pa.signature({'theta':THETA_MAPA_KSN,'A0_m2':A0_GEORGE,
        'limites':{str(k):list(v) for k,v in LIMITES_TRAMOS_M.items()},
        'min_nodos':MIN_NODOS_GEORGE,'min_longitud_m':MIN_LONGITUD_GEORGE_M})
    if firma_actual != gf_firma_ajustes:
        raise ValueError('Cambiaste parámetros o límites: recalcular la sección 5.')
    filas,vistos = [],set()
    for registro in ASOCIACIONES_REVISADAS:
        faltan = {'ID','id_kp','falla','x_ref_m','y_ref_m','motivo'}-set(registro)
        if faltan: raise ValueError(f'Faltan campos en asociación: {sorted(faltan)}')
        rid,falla = str(registro['ID']),str(registro['falla'])
        clave = (falla,rid,str(registro['id_kp']))
        if clave in vistos: raise ValueError(f'Asociación duplicada: {clave}')
        vistos.add(clave)
        trazas = gf_trazas_mapa.loc[gf_trazas_mapa.familia.eq(falla)]
        if trazas.empty or falla not in AZIMUT_RUMBO_GRADOS:
            raise ValueError(f'Falla no configurada: {falla}')
        az = float(AZIMUT_RUMBO_GRADOS[falla])
        if not np.isfinite(az): raise ValueError('Azimut no finito.')
        u = np.array([np.sin(np.deg2rad(az)),np.cos(np.deg2rad(az))])
        partes = [part for geom in trazas.geometry for part in sw._partes_lineales(geom)]
        intervalos = [(float(np.min(np.asarray(part.coords)[:,:2]@u)),
                       float(np.max(np.asarray(part.coords)[:,:2]@u))) for part in partes]
        origen = min(a for a,b in intervalos)
        xy = np.array([registro['x_ref_m'],registro['y_ref_m']],dtype=float)
        if not np.isfinite(xy).all() or not str(registro['motivo']).strip():
            raise ValueError('Referencia no finita o sin justificación.')
        q = tabla_ksn_bajo_KP.loc[tabla_ksn_bajo_KP.ID.eq(rid)
            & tabla_ksn_bajo_KP.id_kp.eq(str(registro['id_kp']))]
        if len(q)!=1 or not np.isfinite(q.iloc[0].get('ksn_040',np.nan)):
            raise ValueError(f'No hay un ajuste positivo único para {clave}.')
        fila = q.iloc[0].to_dict()
        fila.update(falla=falla,distancia_rumbo_km=float((xy@u-origen)/1000),
            x_ref_m=float(xy[0]),y_ref_m=float(xy[1]),motivo=registro['motivo'],
            distancia_a_traza_km=float(trazas.geometry.distance(Point(*xy)).min()/1000))
        filas.append(fila)
        for a,b in intervalos:
            item = (falla,(a-origen)/1000,(b-origen)/1000)
            if item not in gf_alcances: gf_alcances.append(item)
    gf_comparacion = pd.DataFrame(filas)
    display(gf_comparacion.reindex(columns=columnas).sort_values(['falla','distancia_rumbo_km']).round(3))

In [ ]:
vf.rumbo(vf_vista,gf_comparacion,gf_alcances)
gf_firma_asociaciones = gf_pa.signature({'asociaciones':ASOCIACIONES_REVISADAS,
    'azimut':AZIMUT_RUMBO_GRADOS,'firma_ajustes':gf_firma_ajustes})

## 8 Guardar tablas y figuras

Los interruptores `GUARDAR_PRODUCTOS` y `GUARDAR_FIGURAS` están al comienzo. Una ejecución crea una carpeta nueva v1, v2, etc.; no reemplaza entradas ni resultados compartidos. Los perfiles exportados distinguen `ksn_local`, `ksn_regional` y `chi_regional_m`.

Si se modificaron parámetros, límites o asociaciones, se exige recalcular la sección correspondiente antes de exportar. Las figuras guardadas son las producidas desde la última preparación de la escala de color. Los mapas se exportan como HTML y los paneles como PNG.

In [ ]:
if GUARDAR_PRODUCTOS or GUARDAR_FIGURAS:
    sw.validar_ksn(gf_ksn_perfiles,VENTANA_MAPA_KSN_M,THETA_MAPA_KSN)
    firma_ajustes_actual = gf_pa.signature({'theta':THETA_MAPA_KSN,'A0_m2':A0_GEORGE,
        'limites':{str(k):list(v) for k,v in LIMITES_TRAMOS_M.items()},
        'min_nodos':MIN_NODOS_GEORGE,'min_longitud_m':MIN_LONGITUD_GEORGE_M})
    if firma_ajustes_actual != gf_firma_ajustes:
        raise ValueError('Cambiaste el ajuste: recalcular desde sección 6.')
    firma_asoc_actual = gf_pa.signature({'asociaciones':ASOCIACIONES_REVISADAS,
        'azimut':AZIMUT_RUMBO_GRADOS,'firma_ajustes':gf_firma_ajustes})
    if firma_asoc_actual != gf_firma_asociaciones:
        raise ValueError('Cambiaste asociaciones: recalcular sección 7.')
    meta_esperada = {'theta':THETA_MAPA_KSN,'ventana_m':VENTANA_MAPA_KSN_M,
        'rios':gf_ids_mapa,'mostrar_maule':MOSTRAR_MAULE}
    if dict(ksni_fig.layout.meta or {}) != meta_esperada:
        raise ValueError('Recrear mapa en sección 4.')
    if not np.isclose(vf_vista['theta'],THETA_MAPA_KSN) or not np.isclose(vf_vista['ventana_m'],VENTANA_MAPA_KSN_M):
        raise ValueError('Repreparar escala visual en sección 3.')
    carpeta_salida = pa.nueva_carpeta(CARPETA_PRODUCTOS)
    fecha = carpeta_salida.name
    if GUARDAR_PRODUCTOS:
        tablas = {'ksn_bajo_KP':tabla_ksn_bajo_KP,'QC_local':gf_tabla_qc,
            'KP_verificados':gf_manuales,'KP_rechazados':gf_rechazados,'comparacion_rumbo':gf_comparacion}
        for etiqueta,tabla in tablas.items():
            tabla.to_csv(carpeta_salida/f'{etiqueta}_{fecha}.csv',index=False,encoding='utf-8-sig')
        for rid,p in gf_ksn_perfiles.items():
            p.to_csv(carpeta_salida/f'perfil_ksn_R{int(rid):02d}_{fecha}.csv',index=False,encoding='utf-8-sig')
            d = gf_ksn_segmentos[rid]
            pd.DataFrame({'s0_m':d['s0_m'],'s1_m':d['s1_m'],
                'x0_m':d['xy'][:,0,0],'y0_m':d['xy'][:,0,1],
                'x1_m':d['xy'][:,1,0],'y1_m':d['xy'][:,1,1],
                'ksn_local':d['ksn_040'],'ventana_parcial':d['ventana_parcial'],
                'ventana_cruza_KP':d['ventana_cruza_KP']}).to_csv(
                    carpeta_salida/f'segmentos_ksn_R{int(rid):02d}_{fecha}.csv',index=False,encoding='utf-8-sig')
    if GUARDAR_FIGURAS:
        mapas = {'mapa_completo':ksni_fig}
        if vf_mapa_bajo_kp is not None: mapas['mapa_bajo_KP'] = vf_mapa_bajo_kp
        carpeta_figuras = vf.guardar(vf_vista,carpeta_salida/'figuras',mapas=mapas)
    entradas = gf_pa.dependency('perfiles')
    entradas[str(Path(gf_pa.CFG['kp_manual_csv']))] = gf_pa.file_hash(gf_pa.CFG['kp_manual_csv'])
    for clave in ['chaf'] + (['maule'] if MOSTRAR_MAULE else []):
        entradas[gf_pa.CFG[clave]] = gf_pa.input_hash(gf_pa.CFG[clave])
    parametros_color = {k:v for k,v in vf_vista.items() if k!='figuras'}
    info = {'fecha':fecha,'notebook':'04_ksn_Fallas.ipynb',
        'notebook_sha256':gf_pa.file_hash(BASE/'04_ksn_Fallas.ipynb'),
        'corrida_entradas':gf_pa.RUN_ID,'theta':THETA_MAPA_KSN,'ventana_m':VENTANA_MAPA_KSN_M,
        'min_nodos':MIN_NODOS_MAPA_KSN,'A0_m2':A0_GEORGE,'entradas':entradas,
        'ruta_dem':gf_pa.CFG['dem'],'color':parametros_color,
        'huellas_perfiles':{rid:gf_pa.profile_hash(p) for rid,p in gf_perfiles.items()},
        'limites_tramos_m':{str(k):list(v) for k,v in LIMITES_TRAMOS_M.items()},
        'asociaciones_revisadas':ASOCIACIONES_REVISADAS,'azimut_grados':AZIMUT_RUMBO_GRADOS,
        'modulos_sha256':{str(p):gf_pa.file_hash(p) for p in [MODULO_VISUAL,MODULO_GRAFICOS,
            CARPETA_CODIGO/'memoria_comun.py']}}
    (carpeta_salida/f'manifiesto_{fecha}.json').write_text(json.dumps(info,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    print('Productos guardados en:',carpeta_salida)
else:
    print('Exportación desactivada; activar los interruptores cuando se quiera guardar.')

## Relación con los otros análisis
El swath interactivo y la calibración K/U/τ están en **03_K_U_tau_20261019.ipynb**. Aquí se conserva únicamente el análisis de los cauces seleccionados y sus ajustes bajo KP para comparar por rumbo de falla.

Los límites de tramo de este notebook son diagnósticos de ksn bajo KP; no recalibran silenciosamente K de 03, cuyo tramo es B–P_T. La red completa del polígono y sus perfiles χ/litología están en **05_Red_completa_fallas_20261019.ipynb**.

El ksn local de 04 procede de las sensibilidades de 01 (pendiente local S multiplicada por A^θ, ventana base del catálogo). El ksn de la red de 05 usa ajuste z–χ en ventana de longitud de cauce. Deben identificarse método y ventana al comparar mapas; no son estimadores numéricamente idénticos en una ventana finita.